# VOS → Wan2.2 Animate Replacement (sem API de inferência)
O Colab gratuito T4 16 GB NÃO foi validado. O diagnóstico deve passar antes de baixar pesos. Uma cena por ZIP, quadros originais.

In [ ]:
import sys, pathlib, shutil, subprocess
from google.colab import files
print('Espaco livre:',round(shutil.disk_usage('/content').free/2**30,2),'GiB')

## 1. Enviar ZIP VOS (uma cena, FPS original) e imagem do novo personagem.

In [ ]:
print('Escolha ZIP do VOS')
u=files.upload()
ZIP=pathlib.Path('/content')/next(n for n in u if n.lower().endswith('.zip'))
print('Escolha imagem do novo personagem')
u=files.upload()
REF=pathlib.Path('/content')/next(n for n in u if n.lower().endswith(('.png','.jpg','.jpeg')))
print('Selecionados:',ZIP,REF)

## 2. Preparar cena (sem IA). Baixa somente o script do seu repositório GitHub.

In [ ]:
import urllib.request
BRIDGE=pathlib.Path('/content/wan_bridge.py')
urllib.request.urlretrieve('https://raw.githubusercontent.com/aruanmf446-hub/Vos/main/frame_studio/wan_bridge.py',str(BRIDGE))
WORK=pathlib.Path('/content/vos_wan_job')
subprocess.run([sys.executable,str(BRIDGE),'prepare','--zip',str(ZIP),'--reference',str(REF),'--work',str(WORK)],check=True)

In [ ]:
from IPython.display import display, Video
display(Video(str(WORK/'driving.mp4'),embed=True))

## 3. Bloqueio obrigatório de GPU/RAM/disco; só prossiga se passar. 32/24/85 GiB são limiares de segurança, não garantia de execução.

In [ ]:
import torch, psutil, shutil
if not torch.cuda.is_available(): raise RuntimeError('CUDA indisponivel no Colab gratuito')
p=torch.cuda.get_device_properties(0)
vram=p.total_memory/2**30
ram=psutil.virtual_memory().total/2**30
disk=shutil.disk_usage('/content').free/2**30
print({'gpu':p.name,'vram_gib':round(vram,1),'ram_gib':round(ram,1),'free_disk_gib':round(disk,1)})
if vram<32 or ram<24 or disk<85: raise RuntimeError('Recursos insuficientes para tentativa conservadora Wan Animate 14B. Nenhuma inferencia foi executada.')
print('Diagnostico passou (nao garante que Wan conclua).')

## 4. Instalar Wan oficial e baixar pesos via ModelScope, jamais Hugging Face. Download de arquivos via internet; não é API de inferência.

In [ ]:
WAN=pathlib.Path('/content/Wan2.2')
if not WAN.exists():subprocess.run(['git','clone','--depth','1','https://github.com/Wan-Video/Wan2.2.git',str(WAN)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(WAN/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','modelscope'],check=True)
CKPT=pathlib.Path('/content/Wan2.2-Animate-14B')
subprocess.run(['modelscope','download','Wan-AI/Wan2.2-Animate-14B','--local_dir',str(CKPT)],check=True)

## 5. Executar Wan2.2 Animate em Replacement; se der OOM não há resultado.

In [ ]:
subprocess.run([sys.executable,str(BRIDGE),'run','--work',str(WORK),'--wan',str(WAN),'--checkpoint',str(CKPT)],check=True)

## 6. Voltar ao contrato VOS, gerar ZIP editado e baixar.

In [ ]:
OUT=pathlib.Path('/content/VOS_EDITADO.zip')
subprocess.run([sys.executable,str(BRIDGE),'finish','--work',str(WORK),'--output',str(OUT)],check=True)
files.download(str(OUT))

## 7. Retorne ao mesmo VOS no Render
Importar edições > Anexar ZIP > Conferir validação > Aplicar pacote > Montar vídeo (FPS original).